# 02 - Bronze Ingestion: MongoDB master data

Pulls `customer_master` and `product_master` out of MongoDB Atlas
(database `SPT_Stores`) into Bronze Delta tables.

- `customer_master` is loaded **incrementally** (only new/changed docs
  since the last run) because Customer is the SCD2 dimension downstream —
  an overwrite every run would erase the history SCD2 is meant to keep.
- `product_master` is a **full load** every run — there's no history
  requirement for products, so a full overwrite is simplest and still
  correct.

`pymongo` isn't preinstalled on the cluster, so the first cell installs it
with `%pip` — that's a cluster-scoped install for this notebook's session,
not anything running on a laptop.

In [0]:
# Databricks notebook source
%pip install pymongo certifi

In [0]:
dbutils.widgets.text("catalog", "maven_test")
catalog = dbutils.widgets.get("catalog")
database_name = "SPT_Stores"

print(f"Running Bronze Mongo ingestion for catalog: {catalog}")

### Connect to Atlas

The connection string is a secret, never hardcoded — it comes from the
Key Vault-backed secret scope `captone_scope`.

In [0]:
import pymongo
import certifi

mongo_uri = dbutils.secrets.get(scope="captone_scope", key="mongodb-conn-string")
client = pymongo.MongoClient(mongo_uri, tls=True, tlsCAFile=certifi.where())
db = client[database_name]

collections = db.list_collection_names()
print("Collections found:", collections)

### Find the customer and product collections

We match by name instead of hardcoding the exact collection name, in case
it's not spelled exactly `customer_master` / `product_master` in Atlas.

In [0]:
def find_collection(keyword):
    for name in collections:
        if keyword in name.lower():
            return name
    return None

customer_collection = find_collection("Customer_Master")
product_collection = find_collection("Product_Master")
print("customer collection:", customer_collection)
print("product collection:", product_collection)

### Watermark helper (for the incremental customer load)

Reads the last high-water mark we saved, so we only pull documents
changed after that point. If there's no watermark table yet, this is the
first run and we do a full load.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def get_watermark(collection_name):
    table = f"{catalog}.ops.ingestion_watermarks"
    try:
        row = (
            spark.table(table)
            .filter(f"source_name = '{collection_name}'")
            .orderBy(spark.table(table).run_timestamp.desc())
            .limit(1)
            .collect()
        )
    except Exception:
        return None
    return row[0]["watermark_value"] if row else None


def save_watermark(collection_name, watermark_value, rows_ingested):
    table = f"{catalog}.ops.ingestion_watermarks"
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {table} (
            source_name STRING, watermark_value STRING,
            rows_ingested LONG, run_timestamp TIMESTAMP
        ) USING DELTA
    ''')
    spark.sql(f'''
        INSERT INTO {table} VALUES (
            '{collection_name}', '{watermark_value}', {rows_ingested}, current_timestamp()
        )
    ''')

### Ingest one collection

Same function handles both the incremental (customer) and full (product)
loads — the only difference is whether we filter by watermark and whether
we append or overwrite.

In [0]:
def ingest_collection(collection_name, bronze_table, incremental):
    since = get_watermark(collection_name) if incremental else None
    print(f"Loading {collection_name} -> {bronze_table} (incremental={incremental}, since={since})")

    query = {}
    if since is not None:
        query = {"_source_last_modified": {"$gt": since}}

    attempt = 0
    documents = None
    while attempt < 3 and documents is None:
        attempt += 1
        try:
            documents = list(db[collection_name].find(query, {"_id": 0}))
        except Exception as e:
            print(f"  attempt {attempt} failed: {e}")
            if attempt == 3:
                raise

    if not documents:
        print(f"  no new documents for {collection_name}, nothing to write")
        return

    df = spark.createDataFrame(documents) \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_system", lit("MONGODB_ATLAS"))

    if incremental:
        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(bronze_table)
        new_values = [d.get("_source_last_modified") for d in documents if d.get("_source_last_modified")]
        if new_values:
            save_watermark(collection_name, max(new_values), len(documents))
    else:
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)

    print(f"  wrote {len(documents)} rows to {bronze_table}")

In [0]:
if customer_collection is None:
    raise RuntimeError("No collection matching 'customer' found in SPT_Stores")
if product_collection is None:
    raise RuntimeError("No collection matching 'product' found in SPT_Stores")

ingest_collection(customer_collection, f"{catalog}.bronze.bronze_mongodb_customer_master", incremental=True)
ingest_collection(product_collection, f"{catalog}.bronze.bronze_mongodb_product_master", incremental=False)

client.close()
dbutils.notebook.exit("OK")